# RF-DETR: recover trained runs (prediction step only, no retraining)

What happened: in notebook 15 every RF-DETR run trained for the full 100 epochs and saved its weights (`train/last_ema.pth`), then stopped in the prediction step. RF-DETR's head has one extra "no-object" output (index 10); a guard in `train_rfdetr.py` misread it as a 1-based class id and shifted every class id by one. Fixed in the repo: the no-object entries are dropped (as RF-DETR's own evaluator does) and ids are never shifted. Training is unaffected, so the saved weights are valid.

This notebook loads each saved `last_ema.pth`, predicts the val and test images, scores them with the shared evaluator, and writes the usual `metrics.json` + predictions. No training: a few minutes on one GPU.

**Setup (Kaggle):** GPU (T4 x1 or x2), Internet ON, attach the HBFMID dataset, and attach the output of the finished notebook-15 run: right panel **Input -> + Add Input -> Your Work -> Notebooks -> your RF-DETR notebook (notebook 15)**; pick the version that ran the part you want to recover (A, B or C). Then Save Version (Run All).
Download `rfd_rec_results.zip` (small) and paste the printed table to Claude.

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/nithoukhan1/hbfmid.git"
REPO = "/kaggle/working/hbfmid"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rfdetr[train]==1.11.2"], check=True)
import torch
from importlib.metadata import version
print("rfdetr", version("rfdetr"), "| torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU, then restart."
print("repo commit:", subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
assert "no_object_preds_dropped" in open(f"{REPO}/scripts/rfdetr_bench/train_rfdetr.py").read(), "repo is behind: push the RF-DETR fix first"

In [ ]:
import glob, re, zipfile
DATA_ROOT = "/kaggle/input/datasets/nithoukhan2/hbfmid/Human Bone Fractures Multi-modal Image Dataset (HBFMID)/Bone Fractures Detection"
assert os.path.isdir(DATA_ROOT), DATA_ROOT
V2, V1 = f"{REPO}/data_splits_v2", f"{REPO}/data_splits"
OUT_ROOT = "/kaggle/working/results"

# 1) saved weights in the attached notebook output: .../rfd_<PART>/<tag>/train/last_ema.pth
cks = sorted(p for p in glob.glob("/kaggle/input/**/rfd_*/*/train/last_ema.pth", recursive=True))
# 2) if only the results zip was kept, extract just the weights and logs from it
if not cks:
    for zp in glob.glob("/kaggle/input/**/rfd_*_results.zip", recursive=True):
        part = re.search(r"(rfd_[A-Z])_results", zp).group(1); dst = f"/kaggle/working/from_zip/{part}"
        with zipfile.ZipFile(zp) as z:
            for n in z.namelist():
                if n.endswith(("train/last_ema.pth", "train/metrics.csv", "train/training_config.json")) or n.startswith("logs/"):
                    z.extract(n, dst)
    cks = sorted(glob.glob("/kaggle/working/from_zip/rfd_*/*/train/last_ema.pth"))
assert cks, "no last_ema.pth found: attach the notebook-15 output (see the first cell)"

RUNS = []
for ck in cks:
    run_dir = os.path.dirname(os.path.dirname(ck)); part_dir = os.path.dirname(run_dir); part = os.path.basename(part_dir); tag = os.path.basename(run_dir)
    m_off, m_cln = re.match(r"official_rfdetr-medium_s(\d+)_fx$", tag), re.match(r"clean_f(\d)_rfdetr-medium_s(\d+)_fx$", tag)
    if m_off:
        mode, fold, seed, log = "official", 0, int(m_off.group(1)), f"{part_dir}/logs/official_s{m_off.group(1)}_rfdetr-medium.log"
    elif m_cln:
        mode, fold, seed, log = "clean", int(m_cln.group(1)), int(m_cln.group(2)), f"{part_dir}/logs/clean_f{m_cln.group(1)}_s{m_cln.group(2)}_rfdetr-medium.log"
    else:
        print("skipping unknown folder", tag); continue
    RUNS.append((part, tag, mode, fold, seed, part_dir, log))
for r in RUNS:
    print(f"{r[0]:6s} {r[1]:34s} log {'found' if os.path.exists(r[6]) else 'MISSING (train time unknown)'}")
print(len(RUNS), "runs to recover")

In [ ]:
import queue, threading, time
os.makedirs(f"{OUT_ROOT}/logs", exist_ok=True)
q = queue.Queue()
for r in RUNS:
    q.put(r)
failed = []
def worker(gpu):
    while True:
        try:
            part, tag, mode, fold, seed, part_dir, log = q.get_nowait()
        except queue.Empty:
            return
        cmd = [sys.executable, f"{REPO}/scripts/rfdetr_bench/train_rfdetr.py", "--mode", mode, "--fold", str(fold), "--seed", str(seed),
               "--size", "medium", "--data-root", DATA_ROOT, "--splits-dir", V1 if mode == "official" else V2,
               "--out", f"{OUT_ROOT}/{part}", "--predict-only", "--ckpt-root", part_dir, "--log-file", log]
        t0 = time.time()
        with open(f"{OUT_ROOT}/logs/rec_{tag}.log", "w") as f:
            rc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))).returncode
        print(f"[GPU {gpu}] {tag}: {'OK' if rc == 0 else 'FAILED rc=' + str(rc)} in {(time.time() - t0) / 60:.1f} min", flush=True)
        if rc != 0:
            failed.append(tag)
threads = [threading.Thread(target=worker, args=(g,)) for g in range(torch.cuda.device_count())]
[t.start() for t in threads]; [t.join() for t in threads]
print("failed:", failed or "none")
for n in failed:
    print("\n==== tail of", n); print("".join(open(f"{OUT_ROOT}/logs/rec_{n}.log").readlines()[-25:]))

In [ ]:
import zipfile
from pathlib import Path
def zip_small(src, dst):
    """Zip results without checkpoints (*.pth, *.ckpt, *.pt), without the per-run COCO copies and without symlinks."""
    n = 0
    with zipfile.ZipFile(dst, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(Path(src).rglob("*")):
            if p.is_symlink() or p.is_dir() or "coco" in p.relative_to(src).parts or p.suffix in (".pth", ".ckpt", ".pt"):
                continue
            z.write(p, p.relative_to(src)); n += 1
    return n

import json
print(f"{'part':6s} {'run':34s} {'mAP50-95':>9s} {'mAP50':>7s} {'macro':>7s} {'train_min':>9s} {'bg dropped':>10s}  flag")
for p in sorted(glob.glob(f"{OUT_ROOT}/rfd_*/*/metrics.json")):
    r = json.load(open(p)); m = r["test"]["map5095"]; mac = r["test"]["macro_eligible_ap5095"] or 0; ts = r.get("train_seconds")
    print(f"{p.split('/')[-3]:6s} {r['tag']:34s} {m:9.4f} {r['test']['map50']:7.4f} {mac:7.4f} {('%.1f' % (ts / 60)) if ts else 'n/a':>9s} "
          f"{r.get('no_object_preds_dropped_test', '?')!s:>10s}  {'<-- collapsed? check the log' if m < 0.01 else ''}")
n = zip_small(OUT_ROOT, "/kaggle/working/rfd_rec_results.zip")
print("\nzip: /kaggle/working/rfd_rec_results.zip with", n, "files (no checkpoints)")

After the run: download `rfd_rec_results.zip`, unzip once into `results/` (any folder name; Claude renames it), and paste the printed table to Claude. Do NOT unzip the big notebook-15 zip into the repo.